In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q selenium ipywidgets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.4 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from datasets import load_dataset
import sys
from tqdm import tqdm
import json
sys.path.insert(0, "/content/drive/MyDrive/ai_enginner/job_search/AI")

from config import PROMPT

In [4]:
# 데이터셋으로 변환
dataset = load_dataset("json", data_files="/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/chat_data.json", split="train")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 75
})


In [5]:
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 67
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 8
    })
})


In [26]:
# model_loader.py
_model = None
_tokenizer = None
_device = None

def init_model(model_name="skt/A.X-4.0-Light"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch

    _device = "cuda" if torch.cuda.is_available() else "cpu"

    if _device == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_storage=torch.bfloat16,
        )
        _model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            dtype=torch.bfloat16,
            device_map="auto",
        )
    else:
        # CPU 환경: bitsandbytes 없이 로드, float32 사용
        _model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
        )
        _model.to(_device)

    _tokenizer = AutoTokenizer.from_pretrained(model_name)

def get_model():
    return _model, _tokenizer, _device

In [ ]:
init_model('skt/A.X-4.0-Light')
model, tokenizer, device = get_model()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## 데이터셋 토큰화 함수 정의

In [8]:
with open("/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/query_map.json", 'r', encoding='utf-8') as f:
  query_map = json.load(f)

system_content = PROMPT.format(query_field_map=query_map)

def preprocess_function(example):
    # apply_chat_template로 프롬프트 문자열 → 토큰화
    example['messages'] = [{"role": "system", "content": system_content}] + example['messages']

    input_ids = tokenizer.apply_chat_template(
        example['messages'],
        add_generation_prompt=False, # 학습시 False
        return_tensors="pt"
    ).squeeze(0)  # 1차원 텐서로 변환

    return {"input_ids": input_ids}  # attention_mask는 DataCollator가 처리

## 데이터셋 토큰화

In [9]:
train_tokenized_dataset = dataset['train'].map(
    preprocess_function,
    remove_columns=dataset['train'].column_names
)

test_tokenized_dataset = dataset['test'].map(
    preprocess_function,
    remove_columns=dataset['test'].column_names
)

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

## 토큰화 확인

In [10]:
train_tokenized_dataset['input_ids'][0]

[25,
 9,
 239,
 239,
 2093,
 261,
 57,
 54,
 55876,
 239,
 239,
 27010,
 90061,
 32231,
 2204,
 12009,
 41103,
 382,
 13179,
 8379,
 879,
 54,
 239,
 12424,
 1012,
 2539,
 49458,
 5948,
 6451,
 48,
 3245,
 52,
 13099,
 52,
 14519,
 52,
 30874,
 49,
 308,
 1965,
 15894,
 5830,
 12009,
 463,
 24749,
 632,
 52,
 11714,
 30927,
 5681,
 11714,
 6451,
 463,
 33917,
 7678,
 1224,
 54,
 239,
 239,
 5557,
 239,
 239,
 2093,
 261,
 58,
 54,
 1298,
 14495,
 239,
 239,
 53,
 38256,
 10106,
 1079,
 3515,
 2560,
 52,
 13099,
 52,
 14519,
 52,
 1423,
 9883,
 6380,
 12850,
 1936,
 1468,
 54,
 413,
 2155,
 18714,
 3358,
 5948,
 6451,
 804,
 54,
 239,
 53,
 5948,
 6451,
 746,
 50776,
 13317,
 1386,
 8024,
 20884,
 12009,
 329,
 13179,
 1386,
 5885,
 81295,
 3200,
 17074,
 3046,
 1468,
 54,
 239,
 53,
 30874,
 30927,
 591,
 6855,
 42,
 9683,
 463,
 21955,
 5512,
 6380,
 591,
 58,
 52,
 59,
 30057,
 5230,
 42,
 2662,
 591,
 60,
 30057,
 5230,
 42,
 2662,
 3200,
 7381,
 453,
 8092,
 1936,
 1468,
 54,
 239,

In [11]:
decoded_text = tokenizer.decode(train_tokenized_dataset['input_ids'][0], skip_special_tokens=False)
decoded_text

'<|im_start|><|system|>\n\n## 1. Identity\n\n당신은 채용공고 검색을 위한 URL 쿼리 생성 전문가입니다.\n사용자 입력을 분석하여 필수 조건(지역, 직무, 경력, 학력)이 모두 충족되면 URL만 반환하고, 부족한 조건이 있으면 부족한 조건만 직접적으로 요청합니다.\n\n---\n\n## 2. Instructions\n\n- 사용자의 입력 문장에서 지역, 직무, 경력, 학력은 반드시 추출해야 한다. 이 네 가지는 기본 필수 조건이다.\n- 필수 조건 중 하나라도 명확하지 않으면 절대로 URL을 생성하지 말고 사용자에게 추가 질문을 해야 한다.\n- 학력 조건이 "대학" 으로만 주어진 경우에는 반드시 "2,3년제 대학"인지 "4년제 대학"인지 추가 질문으로 구분해야 한다.\n- 추출된 조건은 반드시 Query Field에 존재하는 key와 value로 변환해야 한다.\n- 모든 필수 조건이 완전히 확보된 경우에만 다음 기본 URL에 변환된 쿼리 문자열을 붙여 반환한다:\n  `https://www.saramin.co.kr/zf_user/search?`\n- 응답은 항상 무조건 반드시 완성된 url 또는 부족한 정보를 알려주는 텍스트 2가지로만 응답한다. 그외의 답변은 하지않는다.\n- 부족한 정보에서 사용자가 이해할 수 있는 지역, 직무, 경력, 학력 정보중 부족한것을 알려준다.\n- Query Field에 없는 직무, 지역은 사용자에게 아직 지원하지 않는다고 설명해준다.\n\n\n---\n\n## 3. Examples\n\nExample 1:\nInput: "서울에서 신입 AI 엔지니어 정규직 대졸"\nOutput: "대졸"이 2~3년제 대학졸업인지 4년제 대학교졸업인지 알려주세요.\n\nExample 2:\nInput: "ai 엔지니어 신입"\nOutput: 지역과 학력을 알려주세요.\n\nExample 3:\nInput: "서울 신입 대졸"\nOutput: 직무와 학력 세부사항을 알려주세요. ("대졸"이 2~3년제인지 4년

In [12]:
tokenizer.eos_token

'<|im_end|>'

In [13]:
tokenizer.pad_token

'<|pad|>'

## 데이터콜렉터

In [14]:
from transformers import DataCollatorForLanguageModeling

# 1. 패딩 토큰 설정
tokenizer.pad_token = tokenizer.eos_token

# 2. 라벨, 패딩 처리
data_collator = DataCollatorForLanguageModeling(
		tokenizer=tokenizer,
		mlm=False # 배치 패딩과 attention_mask 자동 처리
)

## 데이터 로더

In [20]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_tokenized_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    test_tokenized_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

In [21]:
from transformers.optimization import Adafactor

model.train()

optimizer = Adafactor(
    model.parameters(),
    lr=None,               # manual 학습률 X
    relative_step=True,    # step마다 자동 조정
    scale_parameter=True,  # 모델 파라미터 크기에 따라 조정
    warmup_init=True       # 초기 스텝에서 학습률 점진적 증가
)

In [22]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        loop = tqdm(dataloader, desc="Evaluating", leave=False)
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

            total_loss += loss.item()
            num_batches += 1
            loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / num_batches
    perplexity = torch.exp(torch.tensor(avg_loss))
    return avg_loss, perplexity.item()

In [23]:
def train_one_epoch(model, dataloader, optimizer, device, epoch):
    model.train()
    total_loss = 0.0
    num_batches = 0

    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=True)
    for batch_idx, batch in enumerate(loop, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / num_batches
    perplexity = torch.exp(torch.tensor(avg_loss))
    return avg_loss, perplexity.item()

In [25]:
num_epochs = 30
all_train_loss, all_train_ppl = [], []
all_val_loss, all_val_ppl = [], []

model.gradient_checkpointing_enable()

for epoch in range(num_epochs):
    train_loss, train_ppl = train_one_epoch(model, train_dataloader, optimizer, device, epoch)
    all_train_loss.append(train_loss)
    all_train_ppl.append(train_ppl)

    # 검증 평가
    val_loss, val_ppl = evaluate(model, test_dataloader, device)
    all_val_loss.append(val_loss)
    all_val_ppl.append(val_ppl)

    print(f"Epoch {epoch+1} | "
          f"Train Loss: {train_loss:.4f}, Train PPL: {train_ppl:.4f} || "
          f"Val Loss: {val_loss:.4f}, Val PPL: {val_ppl:.4f}")

Epoch 1:   0%|          | 0/67 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 120.12 MiB is free. Process 3282 has 14.62 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 444.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import matplotlib.pyplot as plt

# 학습 후 loss & perplexity 시각화
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.plot(range(1, num_epochs+1), all_train_loss, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Epoch-wise Loss Trend")
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(range(1, num_epochs+1), all_train_ppl, marker='o', color='orange')
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.title("Epoch-wise Perplexity Trend")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import os

save_directory = '/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/checkpoint'
os.makedirs(save_directory, exist_ok=True)  # 디렉토리 없으면 생성
model.save_pretrained(save_directory)

In [ ]:
with open("/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/query_map.json", 'r', encoding='utf-8') as f:
  query_map = json.load(f)

system_content = PROMPT.format(query_field_map=query_map)

def inference(user_input):
  model.eval()
  chat = [
      {"role": "system", "content": system_content},
      {"role": "user", "content": user_input}
  ]
  inputs = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt").to(model.device)

  with torch.no_grad():
    output = model.generate(
          inputs,
          max_new_tokens=128,
          do_sample=False,
      )
    len_input_prompt = len(inputs[0])
    response = tokenizer.decode(output[0][len_input_prompt:], skip_special_tokens=True)

  return response

In [ ]:
inference("고졸 신입 ai엔지니어")